# Pipeline de experimentos — Classificação de tweets sobre telecom

-------------------------------------------------------------------
 PIPELINE — CLASSIFICAÇÃO DE TWEETS SOBRE TELECOM
 (modelos clássicos, redes neurais e Transformers | binário e multiclasse)
--------------------------------------------------------------------
Este script único reorganiza os experimentos originalmente feitos em dois notebooks
(`Classificação_twitter_V2.ipynb` e `classificação_twitter_BERTs_V2.ipynb`) em um
pipeline modular, com salvamento padronizado de modelos + metadados para permitir
carregamento automático posterior em um script separado de avaliação externa
(ver `avaliar_modelo_externo.py`).

## PARTE 1 — IMPORTS, CONFIGURAÇÃO CENTRAL E REPRODUTIBILIDADE

Este bloco concentra tudo que precisa ser ajustado para rodar os experimentos em outra
máquina: caminhos, seeds, hiperparâmetros. Nada disso deve aparecer "espalhado" no
restante do script — qualquer número mágico que apareça fora daqui é um bug.

In [1]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import re
import json
import time
import shutil
import string
import pickle
import random
import warnings
import dataclasses
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Optional, Literal, Any

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc,
)
from imblearn.over_sampling import SMOTE

from gensim.models import Word2Vec
from sentence_transformers import SentenceTransformer
import fasttext
import fasttext.util

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model as keras_load_model
from tensorflow.keras.layers import (
    Dense, Embedding, Dropout, SpatialDropout1D,
    Conv1D, GlobalMaxPooling1D, GlobalAveragePooling1D,
    LSTM, GRU, SimpleRNN, Bidirectional,
)

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

warnings.filterwarnings("ignore")

### CONFIGURAÇÃO CENTRAL

In [3]:
# os.environ["TWITTER_EXP_BASE_DIR"] = "/caminho/completo/para/sua/pasta"
os.environ["TWITTER_EXP_BASE_DIR"] = r"D:\Documentos\ITA\11 - CODIGOS MESTRADO\Classificacao Twitter V3"

In [4]:
@dataclass
class Config:
    # --- caminhos -----------------------------------------------------------------
    # variável de ambiente opcional para não obrigar a editar o código-fonte.
    base_dir: Path = field(default_factory=lambda: Path(
        os.environ.get("TWITTER_EXP_BASE_DIR", Path.cwd())
    ))
    data_path: Optional[Path] = None          # setado em __post_init__
    models_dir: Optional[Path] = None
    results_dir: Optional[Path] = None
    fasttext_model_path: Optional[Path] = None

    # --- reprodutibilidade -----------------------------------------------------
    random_seed: int = 42

    # --- splits: 80/10/10 via 80/20 -> 50/50 -------------------------------------
    test_size_holdout: float = 0.20   # separa treino de (val+test)
    val_test_split: float = 0.50      # divide o restante em val/test

    # --- pré-processamento textual ----------------------------------------------
    max_features_vectorizer: int = 2000
    ngram_range: tuple = (1, 2)

    # --- embeddings ---------------------------------------------------------------
    word2vec_dim: int = 150
    word2vec_window: int = 5
    word2vec_min_count: int = 2
    sbert_model_name: str = "paraphrase-multilingual-MiniLM-L12-v2"

    # --- redes neurais (sobre tokens) ----------------------------------------------
    max_words_vocab: int = 10000
    embedding_dim: int = 64
    nn_epochs: int = 30
    nn_batch_size: int = 64

    # --- redes neurais (sobre embeddings pré-computados) ---------------------------
    nn_embed_epochs: int = 30
    nn_embed_batch_size: int = 64

    # --- transformers ---------------------------------------------------------------
    transformer_max_length: int = 128
    transformer_epochs: int = 4
    transfer_learning_batch_size: int = 32
    transfer_learning_lr: float = 1e-3
    fine_tuning_batch_size: int = 8
    fine_tuning_lr: float = 2e-5
    fine_tuning_unfrozen_layers: int = 2      # últimas N camadas do encoder descongeladas
    few_shot_samples_per_class: tuple = (100, 50)

    transformer_models: dict = field(default_factory=lambda: {
        "bert": "bert-base-multilingual-cased",
        "bertweet": "vinai/bertweet-base",
        "betweetbr": "melll-uff/bertweetbr",
        "twitter_roberta": "cardiffnlp/twitter-roberta-base",
        "bertimbau": "neuralmind/bert-base-portuguese-cased",
        "bertabaporu": "pablocosta/bertabaporu-base-uncased",
    })

    # --- SMOTE (bloco de balanceamento, só multiclasse) -----------------------------
    smote_k_neighbors: int = 5

    def __post_init__(self):
        self.base_dir = Path(self.base_dir)
        if self.data_path is None:
            self.data_path = self.base_dir / "dados" / "BD_twitter.csv"
        else:
            self.data_path = Path(self.data_path)
        self.models_dir = self.base_dir / "models"
        self.results_dir = self.base_dir / "results"
        if self.fasttext_model_path is None:
            self.fasttext_model_path = self.base_dir / "external_models" / "cc.pt.300.bin"
        else:
            self.fasttext_model_path = Path(self.fasttext_model_path)

        self.models_dir.mkdir(parents=True, exist_ok=True)
        self.results_dir.mkdir(parents=True, exist_ok=True)
        self.fasttext_model_path.parent.mkdir(parents=True, exist_ok=True)


CONFIG = Config()


def set_global_seed(seed: int = CONFIG.random_seed) -> None:
    """
    Fonte única de aleatoriedade controlada no experimento. Cobre:
    - random / numpy: usados no split, SMOTE, sampling do few-shot;
    - tensorflow: inicialização de pesos das redes neurais Keras;
    - torch (+ CUDA quando disponível): inicialização e treino dos Transformers.
    Os splits do sklearn usam `random_state=seed` explicitamente em cada chamada
    (não dependem só do seed global), mantendo a lógica original dos notebooks.
    """
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## PARTE 2 — PRÉ-PROCESSAMENTO, DADOS E CONVENÇÃO DE NOMES / METADADOS

In [5]:
_STOPWORDS_PT = None


def _get_stopwords():
    global _STOPWORDS_PT
    if _STOPWORDS_PT is None:
        try:
            stopwords.words("portuguese")
        except LookupError:
            nltk.download("stopwords")
        _STOPWORDS_PT = set(stopwords.words("portuguese"))
    return _STOPWORDS_PT


def preprocess_full(text: str) -> str:
    """
    Pré-processamento "full": usado para BoW/TF-IDF nos notebooks originais.
    Lowercase -> remove URL/menção/hashtag(mantém palavra)/número/pontuação/stopword.
    Mantido byte-a-byte igual à função `clean_text` original — não alterar sem
    reexecutar todos os experimentos clássicos que dependem dela.
    """
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#(\w+)", r"\1", text)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    words = text.split()
    words = [w for w in words if w not in _get_stopwords()]
    return " ".join(words)


def preprocess_embeddings(text: str) -> str:
    """
    Pré-processamento "leve": usado antes de Word2Vec/FastText/tokenização Keras.
    Mantido igual à função `clean_text_embeddings` original: só lowercase + remove
    URL/menção. Pontuação e stopwords são preservadas de propósito (afetam a
    qualidade dos embeddings/tokens de forma diferente do BoW).
    """
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    return text


def preprocess_none(text: str) -> str:
    """Sem pré-processamento: retorna o texto original como string."""
    return str(text)


PREPROCESSORS = {
    "sem_preprocessamento": preprocess_none,
    "embeddings": preprocess_embeddings,
    "full": preprocess_full,
}

def apply_preprocessing(texts: pd.Series, preprocessing: str) -> pd.Series:
    return texts.apply(PREPROCESSORS[preprocessing])
 

### CARREGAMENTO E SPLIT DOS DADOS

In [6]:
@dataclass
class SplitData:
    X_train: pd.Series
    X_val: pd.Series
    X_test: pd.Series
    y_train: pd.Series
    y_val: pd.Series
    y_test: pd.Series


def load_dataset(config: Config) -> pd.DataFrame:
    """Carrega o CSV e recria as colunas-alvo exatamente como nos notebooks originais."""
    if not config.data_path.exists():
        raise FileNotFoundError(
            f"Dataset não encontrado em {config.data_path}. "
            f"Ajuste `data_path` na Config ou a variável de ambiente TWITTER_EXP_BASE_DIR."
        )

    df = pd.read_csv(config.data_path, sep=";")

    required_cols = {"texto", "class 3"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Dataset precisa ter as colunas {required_cols}; faltando: {missing}")

    if df["texto"].isnull().any():
        n_null = df["texto"].isnull().sum()
        warnings.warn(f"{n_null} linhas com texto nulo encontradas; serão removidas.")
        df = df.dropna(subset=["texto"]).reset_index(drop=True)

    df["problema_binario"] = (df["class 3"] != 0).astype(int)
    df["problemas_telecom"] = df["class 3"].map({1: 0, 2: 1, 3: 2, 4: 3, 5: 4})

    return df


def make_binary_split(df: pd.DataFrame, config: Config) -> SplitData:
    X = df["texto"]
    y = df["problema_binario"]
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=config.test_size_holdout, stratify=y, random_state=config.random_seed
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=config.val_test_split, stratify=y_temp, random_state=config.random_seed
    )
    return SplitData(X_train, X_val, X_test, y_train, y_val, y_test)


def make_multiclass_split(df: pd.DataFrame, config: Config) -> SplitData:
    mask = df["problemas_telecom"].notna()
    X_multi = df.loc[mask, "texto"]
    y_multi = df.loc[mask, "problemas_telecom"].astype(int)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X_multi, y_multi, test_size=config.test_size_holdout,
        stratify=y_multi, random_state=config.random_seed
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=config.val_test_split,
        stratify=y_temp, random_state=config.random_seed
    )
    return SplitData(X_train, X_val, X_test, y_train, y_val, y_test)


def get_splits(task: str, df: pd.DataFrame, config: Config) -> SplitData:
    """Ponto único de acesso aos splits — usado pelas três famílias, garante que
    todas partem exatamente da mesma divisão treino/val/teste para o mesmo `task`."""
    if task == "binario":
        return make_binary_split(df, config)
    if task == "multiclasse":
        return make_multiclass_split(df, config)
    raise ValueError(f"task desconhecido: {task}")

### CONVENÇÃO DE NOMES E METADADOS

In [7]:
Task = Literal["binario", "multiclasse"]
Preprocessing = Literal["sem_preprocessamento", "embeddings", "full"]
ModelFamily = Literal["classicos", "redes_neurais", "transformers"]

"""
Este objeto é o que resolve o requisito: guardar, junto de cada modelo salvo, 
tudo que é necessário para identificá-lo e recarregá-lo sem configuração manual. 
Serializado como <nome>.metadata.json ao lado de cada artefato de modelo.
"""

@dataclass
class ExperimentMetadata:
    task: Task                       # "binario" | "multiclasse"
    preprocessing: Preprocessing              # "sem_preprocessamento" | "embeddings" | "full"
    model_family: ModelFamily
    model_name: str                 # ex.: "SVM", "LSTM", "bertimbau"
    training_strategy: Optional[str] = None  # None p/ clássicos e RNs; obrigatório p/ transformers
    num_labels: int = 2
    embedding: Optional[str] = None         # ex.: "TFIDF", "Word2Vec", "SBERT", "tokens"; None p/ transformers
    balanced: bool = False                  # True só no bloco de balanceamento (SMOTE), multiclasse
    extra: dict = field(default_factory=dict)

    @property
    def experiment_id(self) -> str:
        """<task>__<preprocessing>__<model_family>__<model_name>__<embedding>__<estrategia>[_n<N>][_BAL]"""
        parts = [
            self.task, self.preprocessing, self.model_family, self.model_name,
            self.embedding or "na", self.training_strategy or "na",
        ]
        name = "__".join(parts)
        if self.extra.get("few_shot_n") is not None:
            name += f"_n{self.extra['few_shot_n']}"
        if self.balanced:
            name += "_BAL"
        return name

    def save_dir(self, config: Config) -> Path:
        d = config.models_dir / self.task / self.preprocessing / self.model_family
        d.mkdir(parents=True, exist_ok=True)
        return d

    def to_json(self) -> str:
        return json.dumps(asdict(self), indent=2, ensure_ascii=False)

    @classmethod
    def from_json(cls, text: str) -> "ExperimentMetadata":
        return cls(**json.loads(text))


def check_task_consistency(metadata: ExperimentMetadata, y) -> None:
    """Confere se o número de classes do rótulo bate com o que a metadata declara —
    evita avaliar um modelo binário como multiclasse ou vice-versa."""
    n_classes_observed = len(set(np.asarray(y).tolist()))
    if metadata.task == "binario" and n_classes_observed > 2:
        raise ValueError(
            f"[{metadata.experiment_id}] Modelo binário recebeu rótulos com "
            f"{n_classes_observed} classes distintas — provável dataset multiclasse."
        )
    if metadata.task == "multiclasse" and metadata.num_labels != n_classes_observed:
        warnings.warn(
            f"[{metadata.experiment_id}] Modelo espera {metadata.num_labels} classes, "
            f"dataset tem {n_classes_observed}. Confira se é o dataset certo."
        )

### BALANCEAMENTO VIA class_weight (redes neurais e Transformers)

SMOTE (usado nos clássicos) reamostra o vetor de features — funciona bem em cima
de representações tabulares (TF-IDF, embeddings médios), mas não se aplica bem a
sequências de tokens nem a texto cru (não existe uma forma natural de "interpolar"
duas sequências de tokens). Para redes neurais e Transformers, o equivalente é
ponderar a função de perda por classe (`class_weight`), sem alterar o conjunto de
treino em si — cada família tem sua própria forma de aplicar isso (Keras aceita
`class_weight=` direto no `.fit()`; para os Transformers em PyTorch, o peso entra
manualmente em `nn.CrossEntropyLoss(weight=...)`).

In [8]:
def compute_keras_class_weight(y) -> dict:
    y_arr = np.asarray(y)
    classes = np.unique(y_arr)
    weights = compute_class_weight("balanced", classes=classes, y=y_arr)
    return {int(c): float(w) for c, w in zip(classes, weights)}


def compute_torch_class_weight(y, num_labels: int, device) -> "torch.Tensor":
    y_arr = np.asarray(y)
    classes_presentes = np.unique(y_arr)
    pesos_presentes = compute_class_weight("balanced", classes=classes_presentes, y=y_arr)
    peso_por_classe = {int(c): float(w) for c, w in zip(classes_presentes, pesos_presentes)}
    vetor = [peso_por_classe.get(i, 1.0) for i in range(num_labels)]
    return torch.tensor(vetor, dtype=torch.float32, device=device)

## PARTE 3 — REPRESENTAÇÕES DE TEXTO (vetorização/embeddings) E I/O UNIFICADO DE MODELOS

Cada classe de representação sabe: (1) como se ajustar aos dados de treino,
(2) como transformar qualquer lista de textos novos do mesmo jeito, e
(3) como se serializar/desserializar por completo — incluindo os objetos
"coadjuvantes" (vectorizer, Word2Vec treinado, etc.) 

In [9]:
class BaseRepresentation:
    name: str = "base"

    def fit(self, texts: pd.Series) -> "BaseRepresentation":
        raise NotImplementedError

    def transform(self, texts) -> np.ndarray:
        raise NotImplementedError

    def state_dict(self) -> dict:
        """O que precisa ser serializado para reconstruir esta representação depois."""
        raise NotImplementedError

    @classmethod
    def from_state_dict(cls, state: dict) -> "BaseRepresentation":
        raise NotImplementedError


class SklearnVectorizerRepresentation(BaseRepresentation):
    """BoW (CountVectorizer) ou TF-IDF — ambos seguem a mesma interface do sklearn."""

    def __init__(self, name: str, config: Config):
        self.name = name
        if name == "BagOfWords":
            self.vectorizer = CountVectorizer(
                max_features=config.max_features_vectorizer, ngram_range=config.ngram_range
            )
        elif name == "TFIDF":
            self.vectorizer = TfidfVectorizer(
                max_features=config.max_features_vectorizer, ngram_range=config.ngram_range,
                min_df=2, max_df=0.9,
            )
        else:
            raise ValueError(f"Vetorizador desconhecido: {name}")

    def fit(self, texts):
        self.vectorizer.fit(texts)
        return self

    def transform(self, texts):
        return self.vectorizer.transform(texts)

    def state_dict(self):
        return {"representation_type": "sklearn_vectorizer", "name": self.name, "vectorizer": self.vectorizer}

    @classmethod
    def from_state_dict(cls, state):
        obj = cls.__new__(cls)
        obj.name = state["name"]
        obj.vectorizer = state["vectorizer"]
        return obj


class Word2VecRepresentation(BaseRepresentation):
 
    name = "Word2Vec"

    def __init__(self, config: Config):
        self.config = config
        self.model: Optional[Word2Vec] = None

    def fit(self, texts):
        tokens = [t.split() for t in texts]
        self.model = Word2Vec(
            tokens, vector_size=self.config.word2vec_dim,
            window=self.config.word2vec_window, min_count=self.config.word2vec_min_count,
            workers=4, seed=self.config.random_seed,
        )
        return self

    def transform(self, texts):
        vectors = []
        for text in texts:
            words = text.split()
            vecs = [self.model.wv[w] for w in words if w in self.model.wv]
            vectors.append(np.mean(vecs, axis=0) if vecs else np.zeros(self.model.vector_size))
        return np.array(vectors)

    def state_dict(self):
        return {"representation_type": "word2vec", "model": self.model}

    @classmethod
    def from_state_dict(cls, state):
        obj = cls.__new__(cls)
        obj.model = state["model"]
        return obj


class FastTextRepresentation(BaseRepresentation):
    """
    FastText usa um modelo pré-treinado (cc.pt.300.bin, baixado do repositório oficial
    do Facebook), não um modelo ajustado nos dados do experimento.
 
    A verificação de existência sozinha não é suficiente: um download interrompido
    (conexão instável, disco cheio, etc.) deixa um arquivo presente mas inválido/
    incompleto, e `fasttext.load_model` falha com um erro pouco claro. Por isso
    `fit` também confere um tamanho mínimo plausível antes de tentar carregar, e
    refaz o download automaticamente se o arquivo existente parecer corrompido.
    """
    name = "FastText"
    MIN_VALID_SIZE_MB = 500   # cc.pt.300.bin válido tem ~6GB; bem menos que isso = corrompido
 
    def __init__(self, config: Config):
        self.config = config
        self.model: Optional["fasttext.FastText._FastText"] = None
        self.model_path = config.fasttext_model_path
 
    @classmethod
    def _looks_valid(cls, path: Path) -> bool:
        try:
            return path.is_file() and path.stat().st_size >= cls.MIN_VALID_SIZE_MB * 1024 * 1024
        except OSError:
            return False
 
    def fit(self, texts):
        if self.model_path.is_file() and not self._looks_valid(self.model_path):
            warnings.warn(
                f"Arquivo em {self.model_path} existe mas parece incompleto/corrompido "
                f"(menor que {self.MIN_VALID_SIZE_MB}MB) — removendo para baixar de novo."
            )
            self.model_path.unlink()
 
        if not self.model_path.is_file():
            print(f"Modelo FastText (pt) não encontrado em {self.model_path}. Baixando...")
            cwd = Path.cwd()
            downloaded_name = "cc.pt.300.bin"
            # limpa qualquer resíduo de download anterior no cwd antes de tentar de novo,
            # senão `if_exists="ignore"` reaproveita um arquivo parcial/corrompido
            for leftover in (cwd / downloaded_name, cwd / f"{downloaded_name}.gz"):
                if leftover.exists():
                    leftover.unlink()
 
            fasttext.util.download_model("pt", if_exists="ignore")
 
            downloaded_path = cwd / downloaded_name
            if not downloaded_path.is_file():
                raise RuntimeError(
                    f"Download do FastText (pt) falhou: {downloaded_path} não foi criado. "
                    f"Baixe manualmente em https://fasttext.cc/docs/en/crawl-vectors.html "
                    f"(arquivo 'bin' da linha 'pt') e coloque em {self.model_path}."
                )
            shutil.move(str(downloaded_path), str(self.model_path))
 
            if not self._looks_valid(self.model_path):
                size_mb = self.model_path.stat().st_size / (1024 * 1024)
                self.model_path.unlink()
                raise RuntimeError(
                    f"Download do FastText (pt) ficou incompleto (apenas {size_mb:.1f}MB, "
                    f"esperado vários GB). Verifique sua conexão/espaço em disco e tente de "
                    f"novo, ou baixe manualmente em "
                    f"https://fasttext.cc/docs/en/crawl-vectors.html e coloque em {self.model_path}."
                )
 
        self.model = fasttext.load_model(str(self.model_path))
        return self
 
    def transform(self, texts):
        vectors = []
        for text in texts:
            words = text.split()
            vecs = [self.model.get_word_vector(w) for w in words if w in self.model]
            vectors.append(np.mean(vecs, axis=0) if vecs else np.zeros(self.model.get_dimension()))
        return np.array(vectors)
 
    def state_dict(self):
        return {"representation_type": "fasttext", "model_path": str(self.model_path)}
 
    @classmethod
    def from_state_dict(cls, state):
        obj = cls.__new__(cls)
        obj.model_path = Path(state["model_path"])
        if not obj.model_path.is_file():
            raise FileNotFoundError(
                f"Binário do FastText não encontrado em {obj.model_path}. "
                f"Ele precisa estar presente para recarregar este modelo."
            )
        obj.model = fasttext.load_model(str(obj.model_path))
        return obj


class SBERTRepresentation(BaseRepresentation):

    name = "SBERT"

    def __init__(self, config: Config):
        self.config = config
        self.model_name = config.sbert_model_name
        self.model: Optional[SentenceTransformer] = None

    def fit(self, texts):
        self.model = SentenceTransformer(self.model_name)
        return self

    def transform(self, texts):
        texts = texts.tolist() if isinstance(texts, pd.Series) else list(texts)
        return self.model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

    def state_dict(self):
        return {"representation_type": "sbert", "model_name": self.model_name}

    @classmethod
    def from_state_dict(cls, state):
        obj = cls.__new__(cls)
        obj.model_name = state["model_name"]
        obj.model = SentenceTransformer(obj.model_name)
        return obj


REPRESENTATION_CLASSES = {
    "sklearn_vectorizer": SklearnVectorizerRepresentation,
    "word2vec": Word2VecRepresentation,
    "fasttext": FastTextRepresentation,
    "sbert": SBERTRepresentation,
}

### CLASSIFICADORES CLÁSSICOS 

In [10]:
def build_classifiers() -> dict:
    return {
        "SVM": SVC(kernel="linear", probability=False),
        "NaiveBayes": MultinomialNB(),
        "LogisticRegression": LogisticRegression(max_iter=2000),
        "RandomForest": RandomForestClassifier(n_estimators=300, random_state=CONFIG.random_seed),
        "KNN": KNeighborsClassifier(n_neighbors=3, metric="euclidean"),
    }


def build_vectorizers(config: Config) -> dict:
    return {
        "BagOfWords": lambda: SklearnVectorizerRepresentation("BagOfWords", config),
        "TFIDF": lambda: SklearnVectorizerRepresentation("TFIDF", config),
    }

### AVALIAÇÃO 

In [11]:
def evaluate_predictions(y_true, y_pred, task: Task) -> dict:
    """
    Único lugar do script que calcula accuracy/precision/recall/F1. Escolhe
    `average="binary"` só quando `task == "binario"`; caso contrário
    usa "macro" (para não favorecer classes majoritárias) e também registra o
    "weighted" F1 para referência — replicando o padrão já usado nos blocos
    multiclasse dos notebooks originais. Isso evita reintroduzir o erro descrito
    no item 10 do pedido (usar `average="binary"` num problema multiclasse).
    """
    average = "binary" if task == "binario" else "macro"
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average=average, zero_division=0),
        "recall": recall_score(y_true, y_pred, average=average, zero_division=0),
        "f1": f1_score(y_true, y_pred, average=average, zero_division=0),
    }
    if task == "multiclasse":
        metrics["f1_weighted"] = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    return metrics


def save_error_analysis(texts, y_true, y_pred, metadata: ExperimentMetadata, config: Config) -> None:
    texts = list(texts) if not isinstance(texts, list) else texts
    df_pred = pd.DataFrame({
        "texto": texts,
        "true": np.asarray(y_true),
        "pred": np.asarray(y_pred),
    })
    errors = df_pred[df_pred["true"] != df_pred["pred"]]
    corrects = df_pred[df_pred["true"] == df_pred["pred"]]

    out_dir = config.results_dir / metadata.task / "error_analysis"
    out_dir.mkdir(parents=True, exist_ok=True)
    errors.to_csv(out_dir / f"{metadata.experiment_id}__errors.csv", index=False)
    corrects.to_csv(out_dir / f"{metadata.experiment_id}__corrects.csv", index=False)

### SALVAMENTO / CARREGAMENTO UNIFICADO — MODELOS CLÁSSICOS

In [12]:
def save_classical_model(classifier, representation: BaseRepresentation,
                          metadata: ExperimentMetadata, config: Config) -> Path:
    """
    Salva classificador + representação (com todos os objetos necessários para
    transformar dados novos, ex.: o TfidfVectorizer ajustado) + metadados, tudo
    em um único .pkl via joblib, mais um .metadata.json legível para inspeção
    rápida.
    """
    save_dir = metadata.save_dir(config)
    bundle = {
        "classifier": classifier,
        "representation_state": representation.state_dict(),
        "metadata": asdict(metadata),
    }
    model_path = save_dir / f"{metadata.experiment_id}.pkl"
    joblib.dump(bundle, model_path)

    meta_path = save_dir / f"{metadata.experiment_id}.metadata.json"
    meta_path.write_text(metadata.to_json(), encoding="utf-8")
    return model_path


class LoadedClassicalModel:
    """Wrapper devolvido por `load_classical_model`; expõe `.predict(textos_crus)`."""

    def __init__(self, classifier, representation: BaseRepresentation, metadata: ExperimentMetadata,
                 preprocessor):
        self.classifier = classifier
        self.representation = representation
        self.metadata = metadata
        self.preprocessor = preprocessor

    def predict(self, raw_texts):
        processed = [self.preprocessor(t) for t in raw_texts]
        X = self.representation.transform(processed if not isinstance(processed, pd.Series) else processed)
        return self.classifier.predict(X)

    def predict_proba(self, raw_texts):
        if not hasattr(self.classifier, "predict_proba"):
            raise AttributeError(f"{self.metadata.model_name} não implementa predict_proba.")
        processed = [self.preprocessor(t) for t in raw_texts]
        X = self.representation.transform(processed)
        return self.classifier.predict_proba(X)


def load_classical_model(path: Path) -> LoadedClassicalModel:
    """Dado só o caminho de um .pkl, identifica o tipo de representação 
    salva, reconstrói o objeto (ex.: o TfidfVectorizer já ajustado) 
    e devolve um wrapper pronto para prever em texto cru."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Modelo não encontrado: {path}")

    bundle = joblib.load(path)
    metadata = ExperimentMetadata(**bundle["metadata"])
    rep_state = bundle["representation_state"]
    rep_type = rep_state["representation_type"]
    if rep_type not in REPRESENTATION_CLASSES:
        raise ValueError(f"Tipo de representação desconhecido no bundle: {rep_type}")

    representation = REPRESENTATION_CLASSES[rep_type].from_state_dict(rep_state)
    preprocessor = PREPROCESSORS[metadata.preprocessing]

    return LoadedClassicalModel(bundle["classifier"], representation, metadata, preprocessor)


def save_results(results: list, task: Task, block_name: str, config: Config) -> pd.DataFrame:
    """Consolida os resultados de um bloco de experimentos em CSV. Chamada pelas
    três famílias de forma independente — cada uma grava seu próprio arquivo,
    nomeado pelo bloco que a gerou (ex.: 'classicos__classicos_full.csv')."""
    df = pd.DataFrame(results)
    out_dir = config.results_dir / task
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{block_name}.csv"
    df.to_csv(out_path, index=False)
    print(f"Resultados salvos em: {out_path}")
    return df

## PARTE 4 — REDES NEURAIS (KERAS)

Duas famílias de rede neural existem nos notebooks originais:
  (1) redes sobre sequências de tokens (MLP, SimpleRNN, LSTM, GRU, BiLSTM, CNN)
  (2) MLP sobre embeddings pré-computados (Word2Vec/FastText/SBERT, vetor fixo)
Ambas passam pelas mesmas etapas (preparar dados -> tokenizar/vetorizar -> montar
arquitetura -> treinar -> validar -> avaliar -> salvar), separadas em funções
próprias abaixo para reuso e para não duplicar código de treino.

### TOKENIZAÇÃO (sequências de tokens)

In [13]:
def build_and_fit_tokenizer(X_train: pd.Series, config: Config) -> tuple:
    max_len = int(X_train.apply(lambda x: len(x.split())).quantile(0.95))
    tokenizer = Tokenizer(num_words=config.max_words_vocab, oov_token="<OOV>")
    tokenizer.fit_on_texts(X_train)
    return tokenizer, max_len


def texts_to_padded(tokenizer: Tokenizer, texts: pd.Series, max_len: int) -> np.ndarray:
    seq = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seq, maxlen=max_len)

### ARQUITETURAS 

In [14]:
def _compile(model, task: Task):
    if task == "binaria":
        model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["binary_accuracy"])
    else:
        model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
                       metrics=["sparse_categorical_accuracy"])
    return model


def build_token_architecture(name: str, config: Config, task: Task,
                              max_len: int, num_labels: int):
    out_units = 1 if task == "binaria" else num_labels
    out_activation = "sigmoid" if task == "binaria" else "softmax"
    vocab, dim = config.max_words_vocab, config.embedding_dim

    if name == "MLP":
        if task == "binaria":
            model = Sequential([
                Embedding(vocab, dim), GlobalAveragePooling1D(),
                Dense(8, activation="relu"), Dropout(0.5),
                Dense(4, activation="relu"), Dropout(0.3),
                Dense(out_units, activation=out_activation),
            ])
        else:
            model = Sequential([
                Embedding(vocab, dim), GlobalAveragePooling1D(),
                Dense(64, activation="relu"), Dropout(0.5),
                Dense(32, activation="relu"), Dropout(0.3),
                Dense(out_units, activation=out_activation),
            ])
    elif name == "SimpleRNN":
        model = Sequential([
            Embedding(vocab, dim, input_length=max_len),
            SimpleRNN(8, activation="tanh"),
            Dense(4, activation="relu"),
            Dense(out_units, activation=out_activation),
        ])
    elif name == "LSTM":
        model = Sequential([
            Embedding(vocab, dim, input_length=max_len),
            LSTM(8, activation="tanh"),
            Dense(4, activation="relu"),
            Dense(out_units, activation=out_activation),
        ])
    elif name == "GRU":
        model = Sequential([
            Embedding(vocab, dim, input_length=max_len),
            GRU(8, activation="tanh"),
            Dense(4, activation="relu"),
            Dense(out_units, activation=out_activation),
        ])
    elif name == "BiLSTM":
        model = Sequential([
            Embedding(vocab, dim), SpatialDropout1D(0.5),
            Bidirectional(LSTM(32, return_sequences=True, dropout=0.3, recurrent_dropout=0.3)),
            GlobalMaxPooling1D(),
            Dense(16, activation="relu"), Dropout(0.3),
            Dense(out_units, activation=out_activation),
        ])
    elif name == "CNN":
        model = Sequential([
            Embedding(vocab, dim, input_length=max_len),
            Conv1D(16, 5, activation="relu"),
            GlobalMaxPooling1D(),
            Dense(out_units, activation=out_activation),
        ])
    else:
        raise ValueError(f"Arquitetura de rede neural desconhecida: {name}")

    return _compile(model, task)


def build_mlp_over_embeddings(input_dim: int, task: Task, num_labels: int):
    out_units = 1 if task == "binaria" else num_labels
    out_activation = "sigmoid" if task == "binaria" else "softmax"
    model = Sequential([
        Dense(16, activation="relu", input_shape=(input_dim,)),
        Dropout(0.3),
        Dense(8, activation="relu"),
        Dense(out_units, activation=out_activation),
    ])
    return _compile(model, task)


NEURAL_ARCHITECTURES = ["MLP", "SimpleRNN", "LSTM", "GRU", "BiLSTM", "CNN"]

### TREINO

In [15]:
def train_neural_model(model, X_train_arr, y_train, X_val_arr, y_val,
                        task: Task, config: Config,
                        epochs: int, batch_size: int, 
                        class_weight: Optional[dict] = None) -> dict:
    start = time.time()
    history = model.fit(
        X_train_arr, y_train, epochs=epochs, batch_size=batch_size,
        validation_data=(X_val_arr, y_val), verbose=0, class_weight=class_weight
    )
    train_time = time.time() - start
    return {"model": model, "history": history.history, "train_time": train_time}


def predict_neural(model, X_arr, task: Task) -> np.ndarray:
    probs = model.predict(X_arr, verbose=0)
    if task == "binaria":
        return (probs > 0.5).astype(int).flatten(), probs
    return np.argmax(probs, axis=1), probs

### SALVAMENTO / CARREGAMENTO — REDES NEURAIS

In [16]:
def save_neural_model(model, metadata: ExperimentMetadata, config: Config,
                       tokenizer: Optional[Tokenizer] = None, max_len: Optional[int] = None,
                       representation: Optional[BaseRepresentation] = None) -> Path:
    """
    Padroniza tudo em formato `.keras` (o notebook original misturava `.h5` e
    `.keras` entre os dois blocos de redes neurais — item g da análise).
    Quando a rede é sobre tokens, o Tokenizer + max_len são salvos ao lado (nunca
    eram salvos no notebook original, tornando os modelos de RN literalmente
    impossíveis de recarregar para inferência em dados novos). Quando é sobre
    embeddings pré-computados, a representação (ex.: SBERT) é salva do mesmo jeito
    que para os modelos clássicos.
    """
    save_dir = metadata.save_dir(config)
    model_path = save_dir / f"{metadata.experiment_id}.keras"
    model.save(model_path)

    extra_path = None
    if tokenizer is not None:
        extra_path = save_dir / f"{metadata.experiment_id}.tokenizer.pkl"
        with open(extra_path, "wb") as f:
            pickle.dump({"tokenizer": tokenizer, "max_len": max_len}, f)
    elif representation is not None:
        extra_path = save_dir / f"{metadata.experiment_id}.representation.pkl"
        with open(extra_path, "wb") as f:
            pickle.dump(representation.state_dict(), f)

    meta_path = save_dir / f"{metadata.experiment_id}.metadata.json"
    meta_path.write_text(metadata.to_json(), encoding="utf-8")
    return model_path


class LoadedNeuralModel:
    def __init__(self, model, metadata: ExperimentMetadata, tokenizer=None, max_len=None,
                 representation: Optional[BaseRepresentation] = None, preprocessor=None):
        self.model = model
        self.metadata = metadata
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.representation = representation
        self.preprocessor = preprocessor

    def predict(self, raw_texts):
        processed = [self.preprocessor(t) for t in raw_texts]
        if self.tokenizer is not None:
            X = texts_to_padded(self.tokenizer, pd.Series(processed), self.max_len)
        elif self.representation is not None:
            X = self.representation.transform(processed)
        else:
            raise RuntimeError("Modelo neural carregado sem tokenizer nem representação.")
        preds, _ = predict_neural(self.model, X, self.metadata.task)
        return preds


def load_neural_model(path: Path) -> LoadedNeuralModel:
    """Implementa o item 5 do pedido para a família "rede_neural"."""
    path = Path(path)
    stem_dir = path.parent
    experiment_id = path.stem
    meta_path = stem_dir / f"{experiment_id}.metadata.json"
    if not meta_path.exists():
        raise FileNotFoundError(f"Metadata não encontrada para {path} (esperado {meta_path}).")
    metadata = ExperimentMetadata.from_json(meta_path.read_text(encoding="utf-8"))

    model = keras_load_model(path)

    tok_path = stem_dir / f"{experiment_id}.tokenizer.pkl"
    rep_path = stem_dir / f"{experiment_id}.representation.pkl"
    tokenizer = max_len = representation = None
    if tok_path.exists():
        with open(tok_path, "rb") as f:
            d = pickle.load(f)
        tokenizer, max_len = d["tokenizer"], d["max_len"]
    elif rep_path.exists():
        with open(rep_path, "rb") as f:
            rep_state = pickle.load(f)
        representation = REPRESENTATION_CLASSES[rep_state["representation_type"]].from_state_dict(rep_state)
    else:
        raise FileNotFoundError(
            f"Nem tokenizer nem representação encontrados ao lado de {path} — "
            f"modelo não pode ser usado para inferência em dados novos."
        )

    preprocessor = PREPROCESSORS[metadata.preprocessing_type]
    return LoadedNeuralModel(model, metadata, tokenizer, max_len, representation, preprocessor)

## PARTE 5 — MODELOS TRANSFORMER (TRANSFER LEARNING / FEW-SHOT / FINE-TUNING)

As três estratégias usam o MESMO loop de treino por época (mesma lógica de
forward/backward/otimização), mas diferem em exatamente 3 pontos:
  - quais parâmetros do modelo ficam congelados/treináveis;
  - batch size e learning rate;
  - (só no few-shot) o tamanho do dataset de treino.

Por isso a função de treino é uma só (`_run_transformer_training`), parametrizada
por uma `TransformerStrategyConfig`, e cada estratégia só monta essa config.

In [17]:
TransformerStrategy = Literal["transfer_learning", "few_shot", "fine_tuning"]


@dataclass
class TransformerStrategyConfig:
    strategy: TransformerStrategy
    batch_size: int
    learning_rate: float
    freeze_fn: Any            # função(model) -> None, aplica o congelamento certo
    few_shot_n: Optional[int] = None    # só usado quando strategy == "few_shot"


def _freeze_transfer_learning(model) -> None:
    """Transfer learning: encoder inteiro congelado, só a cabeça de classificação treina."""
    for param in model.base_model.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True


def _freeze_fine_tuning(model, n_unfrozen_layers: int) -> None:
    """Fine-tuning: tudo congelado, exceto as últimas N camadas do encoder + a cabeça."""
    for param in model.parameters():
        param.requires_grad = False
    for layer in model.base_model.encoder.layer[-n_unfrozen_layers:]:
        for param in layer.parameters():
            param.requires_grad = True
    for param in model.classifier.parameters():
        param.requires_grad = True


def make_strategy_config(strategy: TransformerStrategy, config: Config,
                          few_shot_n: Optional[int] = None) -> TransformerStrategyConfig:
    if strategy == "transfer_learning":
        return TransformerStrategyConfig(
            strategy, config.transfer_learning_batch_size, config.transfer_learning_lr,
            _freeze_transfer_learning,
        )
    if strategy == "few_shot":
        # Few-shot reusa exatamente o congelamento e os hiperparâmetros do transfer
        # learning (é o que o notebook original faz — a única diferença é o tamanho
        # do dataset de treino). Deixado explícito aqui em vez de implícito.
        return TransformerStrategyConfig(
            strategy, config.transfer_learning_batch_size, config.transfer_learning_lr,
            _freeze_transfer_learning, few_shot_n=few_shot_n,
        )
    if strategy == "fine_tuning":
        return TransformerStrategyConfig(
            strategy, config.fine_tuning_batch_size, config.fine_tuning_lr,
            lambda m: _freeze_fine_tuning(m, config.fine_tuning_unfrozen_layers),
        )
    raise ValueError(f"Estratégia desconhecida: {strategy}")


def create_few_shot_dataset(X: pd.Series, y: pd.Series, n_per_class: int, seed: int):
    df = pd.DataFrame({"texto": X, "label": y})
    df_small = (
        df.groupby("label", group_keys=False)
        .apply(lambda g: g.sample(n=min(len(g), n_per_class), random_state=seed))
    )
    return df_small["texto"], df_small["label"]


def _run_transformer_training(model_name: str, X_train, y_train, X_val, y_val,
                                num_labels: int, strategy_cfg: TransformerStrategyConfig,
                                config: Config, save_path: Optional[Path] = None,
                                class_weights: Optional["torch.Tensor"] = None) -> dict:
    start_time = time.time()

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
    strategy_cfg.freeze_fn(model)
    model.to(DEVICE)

    loss_fn = nn.CrossEntropyLoss(weight=class_weights)

    train_enc = tokenizer(list(X_train), padding=True, truncation=True,
                           max_length=config.transformer_max_length, return_tensors="pt")
    val_enc = tokenizer(list(X_val), padding=True, truncation=True,
                         max_length=config.transformer_max_length, return_tensors="pt")
    train_labels = torch.as_tensor(np.asarray(y_train), dtype=torch.long)
    val_labels = torch.as_tensor(np.asarray(y_val), dtype=torch.long)

    train_loader = DataLoader(
        TensorDataset(train_enc["input_ids"], train_enc["attention_mask"], train_labels),
        batch_size=strategy_cfg.batch_size, shuffle=True,
    )
    val_loader = DataLoader(
        TensorDataset(val_enc["input_ids"], val_enc["attention_mask"], val_labels),
        batch_size=strategy_cfg.batch_size,
    )

    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=strategy_cfg.learning_rate)

    history = {"train_losses": [], "val_losses": [], "train_acc": [], "val_acc": []}

    for epoch in range(config.transformer_epochs):
        model.train()
        total_loss, train_preds, train_true = 0.0, [], []
        for batch in train_loader:
            input_ids, attn_mask, labels = [b.to(DEVICE) for b in batch]
            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
            outputs.loss.backward()
            optimizer.step()
            total_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_true.extend(labels.cpu().numpy())

        model.eval()
        val_loss_total, val_preds, val_true = 0.0, [], []
        with torch.no_grad():
            for batch in val_loader:
                input_ids, attn_mask, labels = [b.to(DEVICE) for b in batch]
                outputs = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
                val_loss_total += outputs.loss.item()
                preds = torch.argmax(outputs.logits, dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_true.extend(labels.cpu().numpy())

        history["train_losses"].append(total_loss / len(train_loader))
        history["val_losses"].append(val_loss_total / len(val_loader))
        history["train_acc"].append(accuracy_score(train_true, train_preds))
        history["val_acc"].append(accuracy_score(val_true, val_preds))
        print(f"[{model_name} | {strategy_cfg.strategy}] epoch {epoch+1}/{config.transformer_epochs} "
              f"- val_acc={history['val_acc'][-1]:.4f}")

    task = "binaria" if num_labels == 2 else "multiclasse"
    metrics = evaluate_predictions(val_true, val_preds, task)
    metrics["train_time"] = time.time() - start_time

    if save_path is not None:
        save_path.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(save_path)
        tokenizer.save_pretrained(save_path)

    return {
        "history": history, "metrics": metrics,
        "val_true": val_true, "val_preds": val_preds,
    }


def train_transformer(model_key: str, model_name: str, splits: SplitData,
                       task: Task, preprocessing: Preprocessing, strategy: TransformerStrategy,
                       config: Config, few_shot_n: Optional[int] = None,
                       balanced: bool = False) -> dict:
    """
    Ponto de entrada único para treinar um Transformer em qualquer combinação de
    pré-processamento × estratégia. `model_key` é o apelido curto (ex.: "bertimbau"),
    `model_name` é o identificador completo do HuggingFace Hub.
    """
    num_labels = 2 if task == "binaria" else 5
    strategy_cfg = make_strategy_config(strategy, config, few_shot_n=few_shot_n)

    X_train_raw, y_train = splits.X_train, splits.y_train
    extra = {}
    if strategy == "few_shot":
        X_train_raw, y_train = create_few_shot_dataset(splits.X_train, splits.y_train,
                                                       few_shot_n, config.random_seed)
        extra["few_shot_n"] = few_shot_n

    preprocessor = PREPROCESSORS[preprocessing]
    X_train = X_train_raw.apply(preprocessor)
    X_val = splits.X_val.apply(preprocessor)

    class_weights = compute_torch_class_weight(y_train, num_labels, DEVICE) if balanced else None

    metadata = ExperimentMetadata(
        task=task,
        preprocessing=preprocessing,
        model_family="transformers",
        model_name=model_key,
        training_strategy=strategy,
        num_labels=num_labels,
        embedding=None,
        extra=extra,
    )
    save_path = metadata.save_dir(config) / metadata.experiment_id

    result = _run_transformer_training(
            model_name, X_train, y_train, X_val, splits.y_val,
            num_labels, strategy_cfg, config, save_path=save_path,
            class_weights=class_weights
    )

    meta_path = metadata.save_dir(config) / f"{metadata.experiment_id}.metadata.json"
    meta_path.write_text(metadata.to_json(), encoding="utf-8")

    save_error_analysis(list(X_val), result["val_true"], result["val_preds"], metadata, config)

    return {"metadata": metadata, **result}

### CARREGAMENTO — TRANSFORMERS

In [18]:
class LoadedTransformerModel:
    def __init__(self, model, tokenizer, metadata: ExperimentMetadata, config: Config):
        self.model = model
        self.tokenizer = tokenizer
        self.metadata = metadata
        self.config = config
        self.model.to(DEVICE)
        self.model.eval()

    def predict(self, raw_texts):
        enc = self.tokenizer(list(raw_texts), padding=True, truncation=True,
                              max_length=self.config.transformer_max_length, return_tensors="pt")
        with torch.no_grad():
            input_ids = enc["input_ids"].to(DEVICE)
            attn_mask = enc["attention_mask"].to(DEVICE)
            logits = self.model(input_ids=input_ids, attention_mask=attn_mask).logits
        return torch.argmax(logits, dim=1).cpu().numpy()


def load_transformer_model(path: Path, config: Config) -> LoadedTransformerModel:
    """
    Carrega direto com as ferramentas do HuggingFace (`from_pretrained`), 
    e valida que a arquitetura salva bate com `num_labels` 
    esperado pela metadata, evitando carregar um Transformer com uma 
    cabeça de classificação incompatível.
    """
    path = Path(path)
    meta_path = path.parent / f"{path.name}.metadata.json"
    if not meta_path.exists():
        raise FileNotFoundError(f"Metadata não encontrada para {path} (esperado {meta_path}).")
    metadata = ExperimentMetadata.from_json(meta_path.read_text(encoding="utf-8"))

    tokenizer = AutoTokenizer.from_pretrained(path)
    model = AutoModelForSequenceClassification.from_pretrained(path)

    if model.config.num_labels != metadata.num_labels:
        raise ValueError(
            f"[{metadata.experiment_id}] Modelo salvo com {model.config.num_labels} labels, "
            f"metadata declara {metadata.num_labels} — arquitetura incompatível com o treino original."
        )

    return LoadedTransformerModel(model, tokenizer, metadata, config)

## PARTE 6 — FAMÍLIA: MODELOS CLÁSSICOS (execução independente)

`executar_classicos()` é o único ponto de entrada desta família — não depende de
nada das famílias de redes neurais ou Transformers (nem importa objetos delas).

Por decisão confirmada explicitamente: as 5 representações (BoW, TF-IDF, Word2Vec,
FastText, SBERT) rodam sob os 3 tipos de pré-processamento, não só sob o
pré-processamento "ideal" para cada uma — isso amplia o desenho experimental
original (que mapeava 1 representação por pré-processamento) para permitir
comparar o efeito do pré-processamento em si, isolado da representação.

In [19]:
def _maybe_smote(X, y, config: Config, balanced: bool):
    if not balanced:
        return X, y
    smote = SMOTE(random_state=config.random_seed, k_neighbors=config.smote_k_neighbors)
    return smote.fit_resample(X, y)


def _classicos_representation_builders(config: Config) -> dict:
    return {
        "BagOfWords": lambda: SklearnVectorizerRepresentation("BagOfWords", config),
        "TFIDF": lambda: SklearnVectorizerRepresentation("TFIDF", config),
        "Word2Vec": lambda: Word2VecRepresentation(config),
        "FastText": lambda: FastTextRepresentation(config),
        "SBERT": lambda: SBERTRepresentation(config),
    }


def _classicos_um_preprocessamento(task: Task, preprocessing: Preprocessing,
                                    splits: SplitData, config: Config,
                                    balanced: bool = False) -> list:
    """Roda as 5 representações × classificadores sob um único pré-processamento.
    Reaproveitada pelas três funções públicas abaixo (uma por pré-processamento)."""
    results = []
    num_labels = 2 if task == "binario" else 5

    preprocessor = PREPROCESSORS[preprocessing]
    X_train_txt = splits.X_train.apply(preprocessor)
    X_val_txt = splits.X_val.apply(preprocessor)

    for rep_name, builder in _classicos_representation_builders(config).items():
        representation = builder()
        start = time.time()
        if rep_name == "SBERT" and preprocessing == "sem_preprocessamento":
            pass
        representation.fit(X_train_txt)
        X_train_vec = representation.transform(X_train_txt)
        X_val_vec = representation.transform(X_val_txt)
        rep_time = time.time() - start

        X_train_fit, y_train_fit = _maybe_smote(X_train_vec, splits.y_train, config, balanced)

        for clf_name, clf in build_classifiers().items():
            if clf_name == "NaiveBayes" and rep_name not in ("BagOfWords", "TFIDF"):
                continue

            start_train = time.time()
            clf.fit(X_train_fit, y_train_fit)
            train_time = time.time() - start_train

            preds_val = clf.predict(X_val_vec)
            metrics = evaluate_predictions(splits.y_val, preds_val, task)
            metrics.update({
                "preprocessing": preprocessing, "representation": rep_name,
                "classifier": clf_name, "train_time": train_time,
                "representation_time": rep_time, "balanced": balanced,
            })
            results.append(metrics)

            metadata = ExperimentMetadata(
                task=task, preprocessing=preprocessing, model_family="classicos",
                model_name=clf_name, embedding=rep_name, num_labels=num_labels,
                balanced=balanced,
            )
            save_classical_model(clf, representation, metadata, config)
            save_error_analysis(list(splits.X_val), splits.y_val, preds_val, metadata, config)

    return results


def classicos_sem_preprocessamento(task: Task, splits: SplitData, config: Config,
                                    balanced: bool = False) -> list:
    return _classicos_um_preprocessamento(task, "sem_preprocessamento", splits, config, balanced)


def classicos_embeddings(task: Task, splits: SplitData, config: Config,
                          balanced: bool = False) -> list:
    return _classicos_um_preprocessamento(task, "embeddings", splits, config, balanced)


def classicos_full(task: Task, splits: SplitData, config: Config,
                    balanced: bool = False) -> list:
    return _classicos_um_preprocessamento(task, "full", splits, config, balanced)

### ORQUESTRAÇÃO DA FAMÍLIA

In [20]:
def executar_classicos_binario(config: Config = CONFIG) -> list:
    df = load_dataset(config)
    splits = get_splits("binario", df, config)
    all_results = []
    for fn in (classicos_sem_preprocessamento, classicos_embeddings, classicos_full):
        print(f"[classicos | binario | {fn.__name__}] iniciando...")
        res = fn("binario", splits, config)
        save_results(res, "binario", f"classicos__{fn.__name__}", config)
        all_results.extend(res)
    return all_results


def executar_classicos_multiclasse(config: Config = CONFIG,
                                    incluir_balanceamento: bool = True) -> list:
    """`incluir_balanceamento=True` roda, adicionalmente, a variante com SMOTE
    aplicado ao treino — replicando o bloco de balanceamento dos notebooks
    originais, que só existia para o problema multiclasse."""
    df = load_dataset(config)
    splits = get_splits("multiclasse", df, config)
    all_results = []
    for fn in (classicos_sem_preprocessamento, classicos_embeddings, classicos_full):
        print(f"[classicos | multiclasse | {fn.__name__}] iniciando...")
        res = fn("multiclasse", splits, config)
        save_results(res, "multiclasse", f"classicos__{fn.__name__}", config)
        all_results.extend(res)

        if incluir_balanceamento:
            print(f"[classicos | multiclasse | {fn.__name__} | balanceado] iniciando...")
            res_bal = fn("multiclasse", splits, config, balanced=True)
            save_results(res_bal, "multiclasse", f"classicos__{fn.__name__}__balanced", config)
            all_results.extend(res_bal)
    return all_results


def executar_classicos(config: Config = CONFIG) -> dict:
    print(f"\n{'='*80}\nFAMÍLIA: MODELOS CLÁSSICOS\n{'='*80}")
    set_global_seed(config.random_seed)
    resultado_binario = executar_classicos_binario(config)
    resultado_multiclasse = executar_classicos_multiclasse(config)
    print("\n[classicos] concluído. Modelos em:", config.models_dir)
    return {"binario": resultado_binario, "multiclasse": resultado_multiclasse}

## PARTE 7 — FAMÍLIA: REDES NEURAIS (execução independente)

`executar_redes_neurais()` é o único ponto de entrada desta família.

Decisão de design: as 6 arquiteturas (MLP, SimpleRNN, LSTM, GRU, BiLSTM, CNN) são arquiteturas de SEQUÊNCIA DE TOKENS — rodam sob os 3 pré-processamentos
(sem_preprocessamento/embeddings/full), cada um gerando seu próprio Tokenizer.
O MLP sobre embeddings pré-computados (Word2Vec/FastText/SBERT) — que nos
notebooks originais já vivia dentro do que era chamado de pré-processamento
"para embeddings" — continua sendo tratado como parte do bloco `rn_embeddings`,
e não é replicado nos blocos `sem_preprocessamento`/`full` (não faz sentido gerar
"Word2Vec sem pré-processamento nenhum" como uma categoria própria, já que
Word2Vec já É uma forma de representação vinda desse pré-processamento leve).

In [21]:
def _rn_arquiteturas_de_token(task: Task, preprocessing: Preprocessing,
                               splits: SplitData, config: Config,
                               balanced: bool = False) -> list:
    """Treina as 6 arquiteturas (MLP, SimpleRNN, LSTM, GRU, BiLSTM, CNN) sobre
    sequências de tokens geradas a partir do texto no pré-processamento indicado."""
    results = []
    num_labels = 2 if task == "binario" else 5

    preprocessor = PREPROCESSORS[preprocessing]
    X_train_txt = splits.X_train.apply(preprocessor)
    X_val_txt = splits.X_val.apply(preprocessor)

    class_weight = compute_keras_class_weight(splits.y_train) if balanced else None

    tokenizer, max_len = build_and_fit_tokenizer(X_train_txt, config)
    X_train_pad = texts_to_padded(tokenizer, X_train_txt, max_len)
    X_val_pad = texts_to_padded(tokenizer, X_val_txt, max_len)

    for arch_name in NEURAL_ARCHITECTURES:
        tf.keras.backend.clear_session()
        model = build_token_architecture(arch_name, config, task, max_len, num_labels)
        train_out = train_neural_model(
            model, X_train_pad, splits.y_train, X_val_pad, splits.y_val,
            task, config, config.nn_epochs, config.nn_batch_size,
            class_weight=class_weight
        )
        preds_val, _ = predict_neural(train_out["model"], X_val_pad, task)
        metrics = evaluate_predictions(splits.y_val, preds_val, task)
        metrics.update({"preprocessing": preprocessing, "architecture": arch_name,
                         "embedding": "tokens", "train_time": train_out["train_time"],
                         "balanced": balanced})
        results.append(metrics)

        metadata = ExperimentMetadata(
            task=task, preprocessing=preprocessing, model_family="redes_neurais",
            model_name=arch_name, embedding="tokens", num_labels=num_labels,
            balanced=balanced
        )
        save_neural_model(train_out["model"], metadata, config, tokenizer=tokenizer, max_len=max_len)
        save_error_analysis(list(splits.X_val), splits.y_val, preds_val, metadata, config)

        hist_dir = config.results_dir / task / "histories"
        hist_dir.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(train_out["history"]).to_csv(hist_dir / f"{metadata.experiment_id}.csv", index=False)

    return results


def _rn_mlp_sobre_embeddings(task: Task, splits: SplitData, config: Config,
                             balanced: bool = False) -> list:
    """MLP treinado sobre embeddings pré-computados (Word2Vec/FastText/SBERT).
    Vive só dentro do bloco 'embeddings' — ver docstring do módulo."""
    results = []
    num_labels = 2 if task == "binario" else 5

    class_weight = compute_keras_class_weight(splits.y_train) if balanced else None

    X_train_txt = splits.X_train.apply(preprocess_embeddings)
    X_val_txt = splits.X_val.apply(preprocess_embeddings)

    embedding_builders = {
        "Word2Vec": lambda: Word2VecRepresentation(config),
        "FastText": lambda: FastTextRepresentation(config),
        "SBERT": lambda: SBERTRepresentation(config),
    }
    for emb_name, builder in embedding_builders.items():
        representation = builder()
        if emb_name == "SBERT":
            representation.fit(splits.X_train)
            X_train_e = representation.transform(splits.X_train)
            X_val_e = representation.transform(splits.X_val)
        else:
            representation.fit(X_train_txt)
            X_train_e = representation.transform(X_train_txt)
            X_val_e = representation.transform(X_val_txt)

        tf.keras.backend.clear_session()
        model = build_mlp_over_embeddings(X_train_e.shape[1], task, num_labels)
        train_out = train_neural_model(
            model, X_train_e, splits.y_train, X_val_e, splits.y_val,
            task, config, config.nn_embed_epochs, config.nn_embed_batch_size,
            class_weight=class_weight
        )
        preds_val, _ = predict_neural(train_out["model"], X_val_e, task)
        metrics = evaluate_predictions(splits.y_val, preds_val, task)
        metrics.update({"preprocessing": "embeddings", "architecture": "MLP",
                         "embedding": emb_name, "train_time": train_out["train_time"],
                         "balanced": balanced})
        results.append(metrics)

        metadata = ExperimentMetadata(
            task=task, preprocessing="embeddings", model_family="redes_neurais",
            model_name="MLP", embedding=emb_name, num_labels=num_labels,
            balanced=balanced
        )
        save_neural_model(train_out["model"], metadata, config, representation=representation)
        save_error_analysis(list(splits.X_val), splits.y_val, preds_val, metadata, config)

    return results


def rn_sem_preprocessamento(task: Task, splits: SplitData, config: Config,
                            balanced: bool = False) -> list:
    return _rn_arquiteturas_de_token(task, "sem_preprocessamento", 
                                     splits, config, balanced)


def rn_embeddings(task: Task, splits: SplitData, config: Config, 
                  balanced: bool = False) -> list:
    return (
        _rn_arquiteturas_de_token(task, "embeddings", splits, config, balanced)
        + _rn_mlp_sobre_embeddings(task, splits, config, balanced)
    )


def rn_full(task: Task, splits: SplitData, config: Config, 
            balanced: bool = False) -> list:
    return _rn_arquiteturas_de_token(task, "full", splits, config, balanced)

### ORQUESTRAÇÃO DA FAMÍLIA

In [22]:
def executar_rn_binario(config: Config = CONFIG) -> list:
    df = load_dataset(config)
    splits = get_splits("binario", df, config)
    all_results = []
    for fn in (rn_sem_preprocessamento, rn_embeddings, rn_full):
        print(f"[redes_neurais | binario | {fn.__name__}] iniciando...")
        res = fn("binario", splits, config)
        save_results(res, "binario", f"rn__{fn.__name__}", config)
        all_results.extend(res)
    return all_results


def executar_rn_multiclasse(config: Config = CONFIG,
                             incluir_balanceamento: bool = True) -> list:
    df = load_dataset(config)
    splits = get_splits("multiclasse", df, config)
    all_results = []
    for fn in (rn_sem_preprocessamento, rn_embeddings, rn_full):
        print(f"[redes_neurais | multiclasse | {fn.__name__}] iniciando...")
        res = fn("multiclasse", splits, config)
        save_results(res, "multiclasse", f"rn__{fn.__name__}", config)
        all_results.extend(res)

        if incluir_balanceamento:
            print(f"[redes_neurais | multiclasse | {fn.__name__} | balanceado (class_weight)] iniciando...")
            res_bal = fn("multiclasse", splits, config, balanced=True)
            save_results(res_bal, "multiclasse", f"rn__{fn.__name__}__balanced", config)
            all_results.extend(res_bal)
    return all_results


def executar_redes_neurais(config: Config = CONFIG) -> dict:
    print(f"\n{'='*80}\nFAMÍLIA: REDES NEURAIS\n{'='*80}")
    set_global_seed(config.random_seed)
    resultado_binario = executar_rn_binario(config)
    resultado_multiclasse = executar_rn_multiclasse(config)
    print("\n[redes_neurais] concluído. Modelos em:", config.models_dir)
    return {"binario": resultado_binario, "multiclasse": resultado_multiclasse}

## PARTE 8 — FAMÍLIA: TRANSFORMERS (execução independente)

`executar_transformers()` é o único ponto de entrada desta família.
Combinação confirmada explicitamente: os 6 modelos rodam sob os 3 pré-processamentos × 3 estratégias de treino (transfer_learning, fine_tuning, few_shot com 2 tamanhos), por classificação — 6 × 3 × 4 × 2 = 144 combinações de treino no total entre binário e multiclasse. É bem mais pesado computacionalmente que a versão anterior (que só testava texto cru); rode um pré-processamento de cada vez se for necessário distribuir entre sessões/máquinas — cada `transformers_*()` abaixo é independente.

In [23]:
def _transformers_um_preprocessamento(task: Task, preprocessing: Preprocessing,
                                       splits: SplitData, config: Config,
                                       balanced: bool = False) -> list:
    results = []
    for model_key, model_name in config.transformer_models.items():
        for strategy in ("transfer_learning", "fine_tuning"):
            print(f"[transformers | {task} | {preprocessing} | {model_key} | {strategy} "
                  f"| balanced={balanced}] iniciando...")            
            out = train_transformer(model_key, model_name, splits, task, preprocessing, strategy, config, balanced=balanced)
            row = dict(out["metrics"])
            row.update({"model": model_key, "strategy": strategy, 
                        "preprocessing": preprocessing, "balanced": balanced})
            results.append(row)

        for n in config.few_shot_samples_per_class:
            print(f"[transformers | {task} | {preprocessing} | {model_key} | few_shot n={n} "
                  f"| balanced={balanced}] iniciando...")
            out = train_transformer(model_key, model_name, splits, task, 
                                    preprocessing, "few_shot", config, 
                                    few_shot_n=n, balanced=balanced)
            row = dict(out["metrics"])
            row.update({"model": model_key, "strategy": "few_shot", "few_shot_n": n,
                        "preprocessing": preprocessing, "balanced": balanced})
            results.append(row)

    return results


def transformers_sem_preprocessamento(task: Task, splits: SplitData, 
                                      config: Config, balanced: bool = False) -> list:
    return _transformers_um_preprocessamento(task, "sem_preprocessamento", 
                                             splits, config, balanced)


def transformers_embeddings(task: Task, splits: SplitData, config: Config, 
                            balanced: bool = False) -> list:
    return _transformers_um_preprocessamento(task, "embeddings", splits, 
                                             config, balanced)


def transformers_full(task: Task, splits: SplitData, config: Config, 
                      balanced: bool = False) -> list:
    return _transformers_um_preprocessamento(task, "full", splits, 
                                             config, balanced)

### ORQUESTRAÇÃO DA FAMÍLIA

In [24]:
def executar_transformers_binario(config: Config = CONFIG) -> list:
    df = load_dataset(config)
    splits = get_splits("binario", df, config)
    all_results = []
    for fn in (transformers_sem_preprocessamento, transformers_embeddings, 
               transformers_full):
        print(f"[transformers | binario | {fn.__name__}] iniciando...")
        res = fn("binario", splits, config)
        save_results(res, "binario", f"transformers__{fn.__name__}", config)
        all_results.extend(res)
    return all_results


def executar_transformers_multiclasse(config: Config = CONFIG,
                                       incluir_balanceamento: bool = True) -> list:
    df = load_dataset(config)
    splits = get_splits("multiclasse", df, config)
    all_results = []
    for fn in (transformers_sem_preprocessamento, transformers_embeddings, transformers_full):
        print(f"[transformers | multiclasse | {fn.__name__}] iniciando...")
        res = fn("multiclasse", splits, config)
        save_results(res, "multiclasse", f"transformers__{fn.__name__}", config)
        all_results.extend(res)

        if incluir_balanceamento:
            print(f"[transformers | multiclasse | {fn.__name__} | balanceado (class_weight)] iniciando...")
            res_bal = fn("multiclasse", splits, config, balanced=True)
            save_results(res_bal, "multiclasse", f"transformers__{fn.__name__}__balanced", config)
            all_results.extend(res_bal)
    return all_results

def executar_transformers(config: Config = CONFIG) -> dict:
    print(f"\n{'='*80}\nFAMÍLIA: TRANSFORMERS\n{'='*80}")
    set_global_seed(config.random_seed)
    resultado_binario = executar_transformers_binario(config)
    resultado_multiclasse = executar_transformers_multiclasse(config)
    print("\n[transformers] concluído. Modelos em:", config.models_dir)
    return {"binario": resultado_binario, "multiclasse": resultado_multiclasse}

## PARTE 9 — CARREGAMENTO AUTOMÁTICO UNIVERSAL

Funciona da mesma forma independentemente de qual família gerou o modelo — o
script de avaliação externa (`avaliar_modelo_externo.py`) não precisa saber se
um dado modelo veio de `executar_classicos()`, `executar_redes_neurais()` ou
`executar_transformers()`; a metadata é auto-suficiente.

In [25]:
def discover_models(config: Config) -> list:
    """Retorna uma lista de tuplas (ExperimentMetadata, caminho_do_artefato_principal),
    varrendo `config.models_dir` inteira — não depende de saber de antemão qual
    família/pré-processamento foi usado em cada modelo."""
    found = []
    for meta_path in config.models_dir.rglob("*.metadata.json"):
        try:
            metadata = ExperimentMetadata.from_json(meta_path.read_text(encoding="utf-8"))
        except Exception as e:
            warnings.warn(f"Metadata inválida em {meta_path}: {e}")
            continue

        experiment_id = meta_path.name.replace(".metadata.json", "")
        if metadata.model_family == "classicos":
            artifact = meta_path.parent / f"{experiment_id}.pkl"
        elif metadata.model_family == "redes_neurais":
            artifact = meta_path.parent / f"{experiment_id}.keras"
        elif metadata.model_family == "transformers":
            artifact = meta_path.parent / experiment_id  # é um diretório HuggingFace
        else:
            warnings.warn(f"model_family desconhecido em {meta_path}: {metadata.model_family}")
            continue

        if not artifact.exists():
            warnings.warn(f"Metadata encontrada mas artefato ausente: {artifact}")
            continue
        found.append((metadata, artifact))
    return found


def load_any_model(metadata: ExperimentMetadata, artifact_path: Path, config: Config):
    """Ponto único de entrada para carregar QUALQUER modelo salvo pelo pipeline,
    de qualquer família — resolve o item 9 do pedido num só lugar."""
    if metadata.model_family == "classicos":
        return load_classical_model(artifact_path)
    if metadata.model_family == "redes_neurais":
        return load_neural_model(artifact_path)
    if metadata.model_family == "transformers":
        return load_transformer_model(artifact_path, config)
    raise ValueError(f"model_family não suportado para carregamento: {metadata.model_family}")


def find_model_by_id(experiment_id: str, config: Config):
    """Atalho: acha e já carrega um modelo pelo nome exato do experiment_id."""
    for metadata, artifact_path in discover_models(config):
        if metadata.experiment_id == experiment_id:
            return load_any_model(metadata, artifact_path, config)
    raise FileNotFoundError(f"Nenhum modelo encontrado com experiment_id='{experiment_id}'")

## PARTE 10 — SELEÇÃO DE EXECUÇÃO

Só uma família roda por vez: nada aqui chama automaticamente as três em sequência.
Para rodar mais de uma família, execute o script/notebook mais de uma vez trocando `FAMILIA_A_EXECUTAR`, ou chame a função da família diretamente (ex.: em outra célula do notebook: `executar_classicos()`).

In [ ]:
FAMILIA_A_EXECUTAR = "transformers"   # "classicos" | "redes_neurais" | "transformers"

_FAMILIAS = {
    "classicos": executar_classicos,
    "redes_neurais": executar_redes_neurais,
    "transformers": executar_transformers,
}

if __name__ == "__main__":
    if FAMILIA_A_EXECUTAR not in _FAMILIAS:
        raise ValueError(
            f"FAMILIA_A_EXECUTAR='{FAMILIA_A_EXECUTAR}' inválida. "
            f"Use uma de: {list(_FAMILIAS)}"
        )
    _FAMILIAS[FAMILIA_A_EXECUTAR](CONFIG)